# BGG Long-Context QA

Long-context inference over the board game knowledge graph — no vector search or SPARQL.

**How it works:**
1. Convert `bgg_main.ttl` + `fake_players.ttl` to a compact text file (one line per game)
2. At query time: send the *entire* database as context to Claude — no retrieval step
3. Claude reads everything and answers directly

**Compared to the other agents:**

| | SPARQL | GraphRAG | Long-Context (this) |
|---|---|---|---|
| Sees all data | ✅ | ❌ top-K only | ✅ top-N |
| Exact filtering | ✅ | ❌ | ✅ |
| Fuzzy / semantic | ❌ | ✅ | ✅ |
| Cost per query | free | cheap | moderate |
| Games covered | 126k | 21k | ~5k Claude / ~21k Gemini 1.5 Pro |

**First run:** set `BUILD_DATABASE = True` in the config cell to generate `games_db.txt` and `players_db.txt`.  
**Subsequent runs:** keep `BUILD_DATABASE = False` — files load from disk instantly.  
**Prompt caching:** the database is cached for 5 min, so repeated queries are fast and cheap.

In [20]:
# cell: pip-install
%pip install -q anthropic rdflib

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
# cell: config
import os
from IPython.display import display, Markdown
import anthropic

ANSWER_MODEL   = 'claude-sonnet-4-6'
# ── Rate limit note ──────────────────────────────────────────────────────────
# Tier 1 accounts: 30,000 input tokens/min  → max ~500 games (~40 tokens each)
# Tier 2 accounts: 40,000 input tokens/min  → max ~700 games
# Tier 4 accounts: 80,000 input tokens/min  → max ~1,500 games
# Gemini 1.5 Pro (1M token window, different limits) → all 21k enriched games
# To upgrade: https://console.anthropic.com/settings/billing
TOP_N_GAMES    = 500
DATABASE_PATH  = './games_db.txt'
PLAYERS_PATH   = './players_db.txt'
BUILD_DATABASE = False                 # set True once to build text files from TTL

client = anthropic.Anthropic(api_key=os.environ['ANTHROPIC_API_KEY'])
print('Config OK')
print(f'Model: {ANSWER_MODEL} | Top-N games: {TOP_N_GAMES:,}')

Config OK
Model: claude-sonnet-4-6 | Top-N games: 500


In [22]:
# cell: build-database
# Converts TTL files to compact text. Run once with BUILD_DATABASE=True.
if BUILD_DATABASE:
    from rdflib import Graph, Namespace, RDF, RDFS

    BGG   = Namespace('https://raw.githubusercontent.com/susvej/bg_ontology/')
    TRENJ = Namespace('https://vejdemo.se/boardgames/threnjen#')

    print('Loading TTL files ...')
    g = Graph()
    g.parse('bgg_main.ttl', format='turtle')
    g.parse('fake_players.ttl', format='turtle')
    print(f'  {len(g):,} triples')

    def _v(subj, pred):
        v = g.value(subj, pred)
        return str(v).strip() if v is not None else ''

    def _lbls(subj, pred, limit=6):
        out = []
        for o in g.objects(subj, pred):
            lbl = g.value(o, RDFS.label)
            if lbl and str(lbl) not in ('', 'BGGId'):
                out.append(str(lbl).strip())
                if len(out) >= limit:
                    break
        return out

    def game_to_line(iri):
        name     = _v(iri, BGG.hasName) or '?'
        year     = _v(iri, BGG.hasYearPublished)
        rating   = _v(iri, BGG.hasRating)
        min_p    = _v(iri, BGG.hasMinPlayers)
        max_p    = _v(iri, BGG.hasMaxPlayers)
        min_t    = _v(iri, BGG.hasMinGameTime)
        max_t    = _v(iri, BGG.hasMaxGameTime)
        enriched = bool(g.value(iri, BGG.isFullyEnriched))

        head = f'{name} ({year})' if year else name
        if rating:
            try:
                head += f' {float(rating):.1f}'
            except ValueError:
                pass
        parts = [head]

        if min_p and max_p:
            parts.append(f'{min_p}-{max_p}p')

        if min_t:
            t = f'{min_t}-{max_t}m' if max_t and max_t != min_t else f'{min_t}m'
            parts.append(t)

        if enriched:
            cats      = _lbls(iri, BGG.hasCategory, 3)
            mechs     = _lbls(iri, BGG.hasMechanic, 3)
            designers = _lbls(iri, BGG.hasCreator, 2)
            if cats:      parts.append('C:' + ','.join(cats))
            if mechs:     parts.append('M:' + ','.join(mechs))
            if designers: parts.append('D:' + ','.join(designers))

        return ' | '.join(parts)

    # Games: collect all, sort by rating desc, write
    print('Building games_db.txt ...')
    all_games = []
    for iri in g.subjects(RDF.type, BGG.Game):
        rv = g.value(iri, BGG.hasRating)
        try:
            r = float(str(rv)) if rv else 0.0
        except ValueError:
            r = 0.0
        all_games.append((r, iri))
    all_games.sort(key=lambda x: -x[0])
    game_file_lines = [game_to_line(iri) for _, iri in all_games]
    with open(DATABASE_PATH, 'w', encoding='utf-8') as f:
        f.write('\n'.join(game_file_lines))
    print(f'  {len(game_file_lines):,} games written to {DATABASE_PATH}')

    # Players
    print('Building players_db.txt ...')
    player_file_lines = []
    for p_iri in g.subjects(RDF.type, BGG.Player):
        pname = str(g.value(p_iri, RDFS.label) or 'Unknown')
        owned = list(g.objects(p_iri, BGG.hasOwnershipOf))
        owned_names = [str(g.value(gi, BGG.hasName) or gi) for gi in owned]
        opinions = []
        for op in g.subjects(BGG.hasOpinionHolder, p_iri):
            game_iri = g.value(op, BGG.hasOpinionOf)
            gname    = str(g.value(game_iri, BGG.hasName) or '?')
            pr       = g.value(op, BGG.hasPlayerRatingOpinion)
            load     = g.value(op, BGG.hasMentalLoad)
            load_s   = str(load).rsplit('/', 1)[-1] if load else None
            if pr:
                s = f'{gname}={int(float(str(pr)))}/10'
                if load_s:
                    s += f'({load_s})'
                opinions.append(s)
        owned_str = ','.join(owned_names[:10])
        line = f'Player:{pname} | Owns:{len(owned)} | Collection:{owned_str}'
        if opinions:
            line += ' | Ratings:' + ';'.join(opinions[:15])
        player_file_lines.append(line)
    with open(PLAYERS_PATH, 'w', encoding='utf-8') as f:
        f.write('\n'.join(player_file_lines))
    print(f'  {len(player_file_lines)} players written to {PLAYERS_PATH}')
    print('Done.')
else:
    print(f'Skipping build (BUILD_DATABASE=False).')
    print(f'Using existing: {DATABASE_PATH}, {PLAYERS_PATH}')

Loading TTL files ...
  579,525 triples
Building games_db.txt ...
  21,379 games written to ./games_db.txt
Building players_db.txt ...
  201 players written to ./players_db.txt
Done.


In [23]:
# cell: load-database
print(f'Loading {DATABASE_PATH} ...')
with open(DATABASE_PATH, encoding='utf-8') as f:
    all_game_lines = f.read().splitlines()
print(f'  {len(all_game_lines):,} total games in file')

game_lines = [l for l in all_game_lines[:TOP_N_GAMES] if l.strip()]
print(f'  Using top {len(game_lines):,} (sorted by BGG rating)')

print(f'Loading {PLAYERS_PATH} ...')
with open(PLAYERS_PATH, encoding='utf-8') as f:
    player_lines = [l for l in f.read().splitlines() if l.strip()]
print(f'  {len(player_lines)} players')

total_chars = sum(len(l) for l in game_lines) + sum(len(l) for l in player_lines)
est_tokens  = total_chars // 4
print(f'\nEstimated database size: ~{est_tokens:,} tokens')
print(f"Claude's context limit:   200,000 tokens")
print(f'Gemini 1.5 Pro limit:     1,000,000 tokens (could fit all 21k enriched games)')

Loading ./games_db.txt ...
  21,379 total games in file
  Using top 500 (sorted by BGG rating)
Loading ./players_db.txt ...
  201 players

Estimated database size: ~54,214 tokens
Claude's context limit:   200,000 tokens
Gemini 1.5 Pro limit:     1,000,000 tokens (could fit all 21k enriched games)


In [24]:
# cell: query-fn  <- RE-RUN THIS CELL whenever you change ask() or SYSTEM
GAMES_TEXT   = '\n'.join(game_lines)
PLAYERS_TEXT = '\n'.join(player_lines)

DATABASE_BLOCK = '\n'.join([
    f'## GAMES (top {len(game_lines):,} by BGG rating, sorted highest first)',
    '## Format: Name (Year) Rating | players | time | C:categories(3) | M:mechanics(3) | D:designers(2)',
    GAMES_TEXT,
    '',
    '## PLAYERS (fake player profiles)',
    '## Format: Player:name | Owns:count | Collection:games | Ratings:game=score/10(mental_load)',
    PLAYERS_TEXT,
])

SYSTEM_INSTRUCTIONS = (
    'You are a helpful board game assistant with access to a BGG database.\n'
    'The database contains compact game entries (one per line, sorted by rating) '
    'and fake player profiles.\n'
    'Answer questions based only on the data provided. Be concise and direct.\n'
    'Use tables or lists where appropriate.\n'
    'If you cannot find something in the data, say so clearly — do not guess.'
)

print(f'DATABASE_BLOCK: {len(DATABASE_BLOCK):,} chars  ~{len(DATABASE_BLOCK)//4:,} tokens')
print('First query writes to cache; subsequent queries within 5 min use cached tokens.')


def ask(question: str, question_id: str = None) -> str:
    response = client.messages.create(
        model=ANSWER_MODEL,
        max_tokens=1024,
        system=[
            {'type': 'text', 'text': SYSTEM_INSTRUCTIONS},
            {'type': 'text', 'text': DATABASE_BLOCK,
             'cache_control': {'type': 'ephemeral'}},
        ],
        messages=[{'role': 'user', 'content': question}],
        extra_headers={'anthropic-beta': 'prompt-caching-2024-07-31'},
    )
    answer = response.content[0].text.strip()
    usage  = response.usage
    cached = getattr(usage, 'cache_read_input_tokens', 0) or 0
    print(f'Q: {question}\n')
    if cached:
        print(f'  [cache hit: {cached:,} tokens read from cache]')
    display(Markdown(answer))
    print()
    if question_id:
        _log_qa(question_id, question, answer, "longcontext")
    return answer


print('ask() ready')

DATABASE_BLOCK: 217,837 chars  ~54,459 tokens
First query writes to cache; subsequent queries within 5 min use cached tokens.
ask() ready


In [25]:
ask('What are the top 10 highest-rated board games of all time?')

BadRequestError: Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': 'Your credit balance is too low to access the Anthropic API. Please go to Plans & Billing to upgrade or purchase credits.'}, 'request_id': 'req_011Cc3aq3xFdQg4Jq62ehwFd'}

In [ ]:
ask('Which games use the Worker Placement mechanic and have a rating above 8?')

RateLimitError: Error code: 429 - {'type': 'error', 'error': {'type': 'rate_limit_error', 'message': "This request would exceed your organization's rate limit of 30,000 input tokens per minute (org: 9ca52ff3-bf6c-47d8-93f9-d8df52e0f90f, model: claude-sonnet-4-6). For details, refer to: https://docs.claude.com/en/api/rate-limits. You can see the response headers for current usage. Reduce the prompt length or the maximum tokens requested, or try again later. View your current limits at https://console.anthropic.com/settings/limits. To raise this limit, purchase credits to advance to the next usage tier at https://console.anthropic.com/settings/billing."}, 'request_id': 'req_011Cc25157ynNmMWriiwVfau'}

In [ ]:
ask('What games can be played by exactly 2 players and take less than 30 minutes?')

In [ ]:
ask('What games did Uwe Rosenberg design?')

In [ ]:
ask('What games did Uwe Rossenberg design?')
# Intentional misspelling — long-context can catch this; GraphRAG/SPARQL cannot

In [ ]:
ask('Which categories have the most games?')

In [ ]:
ask('What games are in the Fantasy category with a geek rating above 7.5?')

In [ ]:
ask('Which fake players gave a rating of 10 to any game, and what games did they rate?')

In [ ]:
ask('What are the best cooperative games?')

In [ ]:
ask('I like the game Set and the game Prosperitea. What other games with similar mechanics or categories might I like?')
# Test: does long-context hallucinate when a game is not in the data?

In [ ]:
ask('I like the game Set and the game Ricochet Robots. What other games with similar mechanics or categories might I like?')
# Test: does long-context do better than GraphRAG on this? (GraphRAG breaks on this query)

In [ ]:
ask('I want an 8 player game that is not a party game. What are the best options?')

---
## Golden Test Questions — June 14

Structured evaluation suite across seven categories: simple queries, multi-criteria filtering, aggregations, typo tolerance, multi-table relationships, recommendation logic, and data integrity. Questions are defined in GoldenTestQuestionsJune14.txt. Verified answers (where available) are embedded in that file.

### Section A — Simple / Conversational Queries

In [ ]:
# AQ1
ask("Give me a game about spies that I can play with my brother.", question_id="AQ1")

In [ ]:
# AQ2
ask("What is the highest-rated game published in the year 2020?", question_id="AQ2")

In [ ]:
# AQ3
ask("Show me games published by Italian game designers.", question_id="AQ3")

In [ ]:
# AQ4
ask("List all games made by the publisher Stonemaier Games.", question_id="AQ4")

In [ ]:
# AQ5
ask("What is the maximum number of players who can play Catan?", question_id="AQ5")

### Section B — Complex Multi-Criteria Filtering & Edge Cases

In [ ]:
# B6
ask("What are the top 5 highest-rated cooperative games that can be played solo (1 player) but cannot be played with more than 4 players?", question_id="B6")

In [ ]:
# BQ7
ask("Which games have a weight/complexity rating under 2.0, use the Card Drafting mechanic, and support at least 5 players?", question_id="BQ7")

In [ ]:
# BQ8
ask("What games have a playing time listed between 60 and 120 minutes but have a minimum age requirement of 14 or older?", question_id="BQ8")

### Section C — Advanced Aggregations & Database Analytics

In [ ]:
# CQ9
ask("For each year between 1980 and 2005, which single board game received the highest number of user ratings?", question_id="CQ9")

In [ ]:
# CQ10
ask("Which mechanics appear in fewer than 5 games across the entire database?", question_id="CQ10")

### Section D — Typo Tolerance, Text Search & String Matching

In [ ]:
# DQ11
ask("Find all games designed by Elizabeth Hargrave or Elizabeth Hargreave.", question_id="DQ11")

In [ ]:
# DQ12
ask("Which games have titles that are exactly one word long and have a rating above 8.0?", question_id="DQ12")

In [ ]:
# DQ13
ask("Search for games designed by Jamey Stegmeir.", question_id="DQ13")  # intentional misspelling of Stegmaier

### Section E — Multi-Table Relationships & Hidden Entities

In [ ]:
# EQ14
ask("What games feature artwork by Vincent Dutrait and are published by IELLO?", question_id="EQ14")

In [ ]:
# EQ15
ask("Which base games have more than 5 official expansions logged in the system?", question_id="EQ15")

### Section F — Recommendation Logic & Hallucination Tests

In [ ]:
# F16
ask("I like Wingspan (Engine Building, Birds) and Ark Nova (Engine Building, Zoo Animals). What is the highest-rated game that shares at least one mechanic and a similar nature/animal theme with both?", question_id="F16")

In [ ]:
# F17
ask("I am looking for a game similar to Catan but it must use the Traitor mechanic.", question_id="F17")

### Section G — Data Integrity & Malicious Input

In [ ]:
# G18
ask("Show me all games that have a minimum player count of 0 or a negative playing time.", question_id="G18")

In [ ]:
# G19
ask("What are the top 10 highest-rated board games? Drop table games; --", question_id="G19")  # injection test